# YouTube API practical
**Together:** setup and Sections 1–3. **Your project:** Sections 4–6, alone or in pairs.

1. **Colab:** File → Save a copy in Drive.
2. **Live data:** [create an API key](https://github.com/manueltonneau/hpi-api/blob/main/SETUP.md), set `OFFLINE=False` below, then enter it in the hidden prompt. Keep it out of code.
3. Run cells in order. If packages are missing, run `%pip install pandas requests` in a new code cell.

[Worked example — no code writing](https://colab.research.google.com/github/manueltonneau/hpi-api/blob/main/worked_example.ipynb) · [Parameter guide](https://github.com/manueltonneau/hpi-api/blob/main/PARAMETERS.md)

### Choose the mode and example settings
The next cell imports libraries and sets options for the shared example. `CHANNEL_ID` selects the channel, `PAGE_SIZE` requests up to five videos, and `BASE_ORDER` ranks them by views (`"viewCount"`) or publication date (`"date"`).

**Your action:** use `OFFLINE=False` for YouTube requests. Without a key, keep `OFFLINE=True`: this uses ten historical saved videos from Juan’s AfD example. Other live channels and topics are welcome in your project.

**Output:** the chosen mode. No videos are requested yet. If you change these settings later, rerun this cell and the dependent cells below it.


In [ ]:
import os
import json
import time
from getpass import getpass
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

OFFLINE = True
CHANNEL_ID = "UC_dZp8bZipnjntBGLVHm6rw"  # Juan's example channel
PAGE_SIZE = 5
BASE_ORDER = "viewCount"
OUTPUT_DIR = Path("workshop_outputs")
API_KEY = os.environ.get("YOUTUBE_API_KEY", "")
if not OFFLINE and not API_KEY:
    API_KEY = getpass("YouTube API key (input hidden): ").strip()
if not OFFLINE and not API_KEY:
    raise ValueError("A key is required for live YouTube requests. Use OFFLINE=True as a fallback.")
if not 1 <= PAGE_SIZE <= 50:
    raise ValueError("Use PAGE_SIZE from 1 to 50.")
print("Mode:", "OFFLINE teaching fixtures" if OFFLINE else "LIVE YouTube API")


### Load the fallback data
The next cell loads ten saved video records and defines `offline_payload()`, which constructs example search and video responses from them.

**Your action:** run unchanged in both modes. With `OFFLINE=True`, later requests use these records without contacting YouTube. In live mode, they are loaded but not used for collection.

**Output:** “Loaded 10 saved records; no network requests.” These are reconstructed teaching responses, not the original API responses. Their observation time is unknown; they are not current counts or a complete channel history.


In [ ]:
# @title Load saved records — run { display-mode: "form" }
FIXTURE = {'provenance': {'description': 'Normalized records reconstructed from displayed saved outputs, not raw API responses. Original exact observation time unknown. Video publication dates are not collection timestamps.', 'source_file': 'youtube_germany_annotated2.ipynb', 'source_cells_zero_based': [12, 14], 'records': 10, 'scope': 'Ten videos printed by the supplied notebook. Coverage and original search completeness have not been verified.'}, 'records': [{'video_id': '8ioWqF4k2fo', 'title': 'Brandmauer weg oder Neuwahlen! PK mit Alice Weidel & Tino Chrupalla - AfD-Fraktion im Bundestag', 'published_at': '2026-05-19T12:58:56Z', 'duration': 'PT23M7S', 'views': 308065, 'likes': 25574, 'comments': 2288, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}, {'video_id': 'eYh7yfnluqs', 'title': 'Interview mit Beatrix von Storch zum Ergebnis des Koalitionsgipfels! - AfD-Fraktion im Bundestag', 'published_at': '2026-05-13T15:01:09Z', 'duration': 'PT17M35S', 'views': 249565, 'likes': 20505, 'comments': 1563, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}, {'video_id': 'Uf9nmuTDesk', 'title': 'Regierungsbefragung mit Lars Klingbeil und Verena Hubertz! - AfD-Fraktion im Bundestag', 'published_at': '2026-05-20T14:50:16Z', 'duration': 'PT17M9S', 'views': 149674, 'likes': 7771, 'comments': 1822, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}, {'video_id': 'Hz6SLP23Nv0', 'title': 'LIVE: Pressekonferenz der AfD-Fraktion - Diese Woche im Bundestag', 'published_at': '2026-05-19T08:26:19Z', 'duration': 'PT1H9M57S', 'views': 147981, 'likes': 11521, 'comments': 1183, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}, {'video_id': 'OU-pD4Tm90w', 'title': 'BUNDESTAG LIVE - 80. Sitzung - AfD-Fraktion im Bundestag', 'published_at': '2026-05-21T21:23:45Z', 'duration': 'PT11H55M1S', 'views': 117253, 'likes': 2224, 'comments': 5, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}, {'video_id': 'HbPnlbTchM8', 'title': 'Dr. Anna Rathert zum Untersuchungsausschuss Nord-Stream! - AfD-Fraktion im Deutschen Bundestag', 'published_at': '2026-05-21T10:47:14Z', 'duration': 'PT4M32S', 'views': 112037, 'likes': 7779, 'comments': 805, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}, {'video_id': '5m7dIsDftmQ', 'title': 'Bernd Baumann knöpft sich die Umfrageverlierer Söder, Maier und Banaszak vor! - AfD-Fraktion', 'published_at': '2026-05-19T09:17:11Z', 'duration': 'PT2M22S', 'views': 103756, 'likes': 7568, 'comments': 299, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}, {'video_id': 'IehYwnYBwZs', 'title': 'BUNDESTAG LIVE - 79. Sitzung - AfD-Fraktion im Bundestag', 'published_at': '2026-05-20T18:51:36Z', 'duration': 'PT6H41M30S', 'views': 98033, 'likes': 2274, 'comments': 25, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}, {'video_id': '40qjQvDzUII', 'title': 'Nordstream: Wir fordern einen Untersuchungsausschuss! Markus Frohnmaier im Interview - AfD-Fraktion', 'published_at': '2026-05-20T16:00:05Z', 'duration': 'PT12M53S', 'views': 81091, 'likes': 7003, 'comments': 695, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}, {'video_id': 'iwycKryn3aw', 'title': 'LIVE: Presseerklärung mit Alice Weidel & Tino Chrupalla - AfD-Fraktion im Bundestag', 'published_at': '2026-05-19T12:58:22Z', 'duration': 'PT23M32S', 'views': 72894, 'likes': 6037, 'comments': 612, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}]}

OFFLINE_PAGE = "offline-page:"

def offline_payload(endpoint, parameters):
    if CHANNEL_ID != "UC_dZp8bZipnjntBGLVHm6rw":
        raise ValueError("Offline data contain only Juan's example channel. Use live mode for another channel.")
    records = FIXTURE["records"]
    if endpoint == "search":
        if parameters.get("channelId") != CHANNEL_ID:
            raise ValueError("The offline cache cannot answer a request for another channel.")
        if parameters.get("type") != "video" or parameters.get("q"):
            raise ValueError("Offline fixtures support video searches without keywords only.")
        order = parameters.get("order", "viewCount")
        if order not in {"date", "viewCount"}:
            raise ValueError("Offline exercise supports date or viewCount ordering.")
        field = "published_at" if order == "date" else "views"
        ranked = sorted(records, key=lambda r: r[field], reverse=True)
        token = parameters.get("pageToken", OFFLINE_PAGE + "0")
        if not token.startswith(OFFLINE_PAGE):
            raise ValueError("Unexpected offline page token.")
        start, size = int(token[len(OFFLINE_PAGE):]), int(parameters["maxResults"])
        selected = ranked[start:start + size]
        payload = {"items": [{"id": {"videoId": r["video_id"]},
                              "snippet": {"title": r["title"]}} for r in selected]}
        if start + size < len(ranked):
            payload["nextPageToken"] = OFFLINE_PAGE + str(start + size)
        return payload
    if endpoint == "videos":
        wanted = parameters.get("id", "").split(",")
        by_id = {r["video_id"]: r for r in records}
        return {"items": [{"id": v,
            "snippet": {"title": by_id[v]["title"], "publishedAt": by_id[v]["published_at"],
                        "channelId": by_id[v]["channel_id"]},
            "statistics": {api: str(by_id[v][local]) for api, local in
                           [("viewCount", "views"), ("likeCount", "likes"), ("commentCount", "comments")]
                           if by_id[v][local] is not None},
            "contentDetails": {"duration": by_id[v]["duration"]}}
            for v in wanted if v in by_id]}
    raise ValueError("This endpoint is not available in offline fixtures.")

print("Loaded", len(FIXTURE["records"]), "saved records; no network requests.")


#### Prepare the request helper
The next cell defines `api_get(endpoint, parameters)`. In live mode it builds the YouTube URL, adds your key, sends a **GET** request with `requests.get()`, checks the response and returns JSON as a Python dictionary. Offline mode uses the fallback responses instead.

**Your action:** run unchanged. **Output:** none expected. This defines a function; it does not collect data yet. The function also keeps request settings and responses for saving later, omitting the key from the saved request log.


In [ ]:
# @title Request helper — run { display-mode: "form" }
RAW_RESPONSES = []
REQUEST_LOG = []

def api_get(endpoint, parameters):
    if OFFLINE:
        return offline_payload(endpoint, parameters)
    if endpoint not in {"search", "videos", "channels", "playlistItems", "commentThreads"}:
        raise ValueError("This workshop exposes only read endpoints.")
    import requests
    safe_parameters = {k: v for k, v in parameters.items() if k != "key"}
    parameters = dict(safe_parameters, key=API_KEY)
    for attempt in range(3):
        requested_at = datetime.now(timezone.utc).isoformat()
        try:
            response = requests.get(
                f"https://www.googleapis.com/youtube/v3/{endpoint}",
                params=parameters, timeout=(5, 25),
            )
        except requests.RequestException as exc:
            raise RuntimeError(f"Network request failed ({type(exc).__name__}); credentials omitted.") from None
        REQUEST_LOG.append({"endpoint": endpoint, "parameters": safe_parameters,
                            "status": response.status_code,
                            "requested_at": requested_at})
        if response.status_code in {429, 500, 502, 503, 504} and attempt < 2:
            retry_after = response.headers.get("Retry-After", "")
            delay = min(float(retry_after), 30) if retry_after.isdigit() else 2 ** attempt
            time.sleep(delay)
            continue
        try:
            payload = response.json()
        except ValueError:
            raise RuntimeError(f"HTTP {response.status_code}: response was not valid JSON.") from None
        if not isinstance(payload, dict):
            raise RuntimeError("Unexpected API response: expected a JSON object.")
        if response.status_code != 200 or "error" in payload:
            reasons = [str(e.get("reason", "unknown"))
                       for e in payload.get("error", {}).get("errors", [])]
            raise RuntimeError(f"API request stopped: HTTP {response.status_code}, reasons={reasons}")
        if not isinstance(payload.get("items"), list):
            raise RuntimeError("Unexpected API response: expected an items list.")
        RAW_RESPONSES.append({"endpoint": endpoint, "parameters": safe_parameters,
                              "received_at": datetime.now(timezone.utc).isoformat(),
                              "payload": payload})
        return payload
    raise RuntimeError("Request did not succeed within the retry budget.")


### 1. Find videos
The next cell calls the `search` endpoint. `channelId` selects the channel, `type="video"` excludes other result types, `order` chooses ranking, and `maxResults` limits this page. `part="snippet"` requests basic descriptive information.

**Your action:** run the shared example unchanged. To compare rankings later, change `BASE_ORDER` in the settings cell and rerun the search and the cells that use its results.

**Output:** the number of returned videos and the first result’s ID and title. Results are in `search_data["items"]`; view counts require the next endpoint.

[Official search parameters](https://developers.google.com/youtube/data/reference/rest/v3/search/list)


In [ ]:
search_params = {
    "part": "snippet", "channelId": CHANNEL_ID,
    "type": "video", "order": BASE_ORDER, "maxResults": PAGE_SIZE,
}
search_data = api_get("search", search_params)
print("Videos on this page:", len(search_data["items"]))
if search_data["items"]:
    first = search_data["items"][0]
    print(json.dumps({"id": first["id"], "snippet": {"title": first.get("snippet", {}).get("title")}},
                     indent=2, ensure_ascii=False))


#### Extract the video IDs
The next cell reads each result’s `id.videoId` and removes duplicate IDs while keeping their order. This is local Python processing, not another API request.

**Your action:** run unchanged. **Output:** `video_ids`, a list of identifiers to send in the metadata request. An empty search produces an empty list.


In [ ]:
video_ids = list(dict.fromkeys(item["id"]["videoId"] for item in search_data["items"]))
print(video_ids)


### 2. Get each video’s metadata and statistics
The next cell calls `videos` with the IDs from the search. `part="snippet,statistics,contentDetails"` requests titles and dates, engagement counts, and details such as duration. The IDs become one comma-separated `id` parameter.

**Your action:** run unchanged and find `statistics.viewCount` in the first result. These are **individual video statistics**; **channel totals** such as subscribers use the separate `channels` endpoint.

**Output:** one example video and any IDs without returned metadata. `[]` means no requested IDs are missing. If the search found nothing, this cell skips the request.

[Official video parameters](https://developers.google.com/youtube/data/reference/rest/v3/videos/list)


In [ ]:
stats_params = {"part": "snippet,statistics,contentDetails", "id": ",".join(video_ids)}
stats_data = api_get("videos", stats_params) if video_ids else {"items": []}
if stats_data["items"]:
    first = stats_data["items"][0]
    print(json.dumps({"id": first["id"], "snippet": {"title": first.get("snippet", {}).get("title")},
                      "statistics": first.get("statistics", {})}, indent=2, ensure_ascii=False))
returned_ids = {item["id"] for item in stats_data["items"]}
print("IDs without metadata:", sorted(set(video_ids) - returned_ids))


### 3. Make a table
The next cell defines `make_table()` and applies it to the video response. It selects fields from nested JSON, converts count strings into numbers, and parses publication dates. Missing counts remain missing rather than becoming zero.

**Your action:** run unchanged. **Output:** `baseline_df`, with one row per returned video; the preview shows ID, title, views and comment count. The full table also contains dates, channel IDs, duration and likes.

**End of the guided example.** We searched for videos, requested details for their IDs, then made a table. Now choose your own question below.


In [ ]:
def make_table(payload):
    rows = [{
        "video_id": item["id"],
        "title": item.get("snippet", {}).get("title"),
        "published_at": item.get("snippet", {}).get("publishedAt"),
        "channel_id": item.get("snippet", {}).get("channelId"),
        "duration": item.get("contentDetails", {}).get("duration"),
        "views": item.get("statistics", {}).get("viewCount"),
        "likes": item.get("statistics", {}).get("likeCount"),
        "comments": item.get("statistics", {}).get("commentCount"),
    } for item in payload["items"]]
    columns = ["video_id", "title", "published_at", "channel_id", "duration", "views", "likes", "comments"]
    frame = pd.DataFrame(rows, columns=columns)
    for field in ["views", "likes", "comments"]:
        frame[field] = pd.to_numeric(frame[field], errors="raise").astype("Int64")
    frame["published_at"] = pd.to_datetime(frame["published_at"], utc=True)
    return frame

baseline_df = make_table(stats_data)
print(baseline_df[["video_id", "title", "views", "comments"]].to_string(index=False, max_colwidth=45))


### 4. Your project
Choose a small descriptive question, then use its matching request template. You can adapt these examples:

| Question | `PROJECT_KIND` | One row represents |
| --- | --- | --- |
| Among ten recent returned videos from a channel, which has the most views? | `channel_videos` | A video, including its statistics |
| Which channels appear most often among twenty results for a chosen search phrase? | `topic_videos` | A video, including its statistics |
| Which of two chosen channels has more reported public videos or subscribers? | `channel_statistics` | A channel’s totals |
| How many replies do twenty collected comments receive? | `video_comments` | A top-level comment, including its reply count |

**Workflow:** find an ID if needed → edit one request template → run the provided collector → describe and save the result.

Limit: **20 results per page, at most two pages** for search/comments. Start with one page. Offline mode only selects from the ten saved videos; other channels, channel totals and comments need live access.

[Request parameters and documentation](https://github.com/manueltonneau/hpi-api/blob/main/PARAMETERS.md) · [More project ideas](https://github.com/manueltonneau/hpi-api/blob/main/PROJECTS.md)


#### Find a channel ID — optional, live only
The next cell turns a channel’s **@handle** into its stable channel ID. A name search can return several candidates, so check the displayed titles.

**Your action:** for a channel project, open its YouTube page, copy the @handle, set `RUN_CHANNEL_DISCOVERY=True`, and edit `CHANNEL_LOOKUP`. Copy the returned ID into your request’s `channelId` (videos) or `id` (channel totals). The finder does not update your request automatically.

**Output:** candidate IDs and channel names. Skip this lookup for a topic search; for comments, use the video ID after `v=` in a watch-page URL instead. Offline mode skips the lookup.


In [ ]:
RUN_CHANNEL_DISCOVERY = False
CHANNEL_LOOKUP = "@GoogleDevelopers"  # Paste a @handle, or enter a channel name.
if RUN_CHANNEL_DISCOVERY and not OFFLINE:
    lookup = CHANNEL_LOOKUP.strip()
    if not lookup:
        raise ValueError("Enter a @handle or channel name.")
    if lookup.startswith("@"):
        candidates = api_get("channels", {"part": "snippet", "forHandle": lookup})
        found = [(item["id"], item["snippet"]["title"]) for item in candidates["items"]]
    else:
        candidates = api_get("search", {"part": "snippet", "type": "channel", "q": lookup, "maxResults": 5})
        found = [(item["id"]["channelId"], item["snippet"]["title"]) for item in candidates["items"]]
    for channel_id, title in found:
        print(channel_id, title)
    if not found:
        print("No match. Check the handle or try a channel name.")
else:
    print("Channel lookup is optional and requires live mode.")


#### Choose and edit ONE request
The next cell contains four templates. `PROJECT_KIND` selects which one is copied into `PROJECT_PARAMS`; the other three are ignored when collecting.

**Your action:** set `PROJECT_KIND` and edit the matching dictionary:
- **Videos and their statistics:** `channelId` for a channel or `q` for a topic; `order="date"` requests recent results and `"viewCount"` ranks by views; `maxResults` is the page size.
- **Channel totals:** `id` is one verified channel ID or several separated by commas.
- **Comments:** `videoId` selects a video; `order="time"` requests recent comments and `"relevance"` uses the API’s relevance ranking.

Keep the supplied `part` and `type` values. Start with `PROJECT_MAX_PAGES=1` and 10–20 results. Optional `publishedAfter`/`publishedBefore` filters use date-time strings such as `2026-01-01T00:00:00Z`.

**Output:** none expected; this only prepares settings. Rerun this cell after each edit, then run the collector and analysis below. [Parameter guide](https://github.com/manueltonneau/hpi-api/blob/main/PARAMETERS.md)


In [ ]:
# Choose and edit ONE of these request templates.
PROJECT_KIND = "channel_videos"  # Or "topic_videos" / "channel_statistics" / "video_comments"
PROJECT_MAX_PAGES = 1           # At most 2; applies to video search and comments

channel_video_request = {
    "part": "snippet", "type": "video", "channelId": CHANNEL_ID,
    "order": "date", "maxResults": 10,
}
topic_video_request = {
    "part": "snippet", "type": "video", "q": "climate policy",
    "order": "date", "maxResults": 10,
}
channel_statistics_request = {
    "part": "snippet,statistics", "id": CHANNEL_ID,
    # For multiple channels use a comma-separated list of verified IDs.
}
comment_request = {
    "part": "snippet", "videoId": str(baseline_df.video_id.iloc[0]) if not baseline_df.empty else "PASTE_VIDEO_ID",
    "order": "time", "maxResults": 20, "textFormat": "plainText",
}
PROJECT_PARAMS = {
    "channel_videos": channel_video_request,
    "topic_videos": topic_video_request,
    "channel_statistics": channel_statistics_request,
    "video_comments": comment_request,
}[PROJECT_KIND].copy()

# Optional video-search edits (live API; offline approximates title/date filtering):
# PROJECT_PARAMS["q"] = "interview"
# PROJECT_PARAMS["publishedAfter"] = "2026-01-01T00:00:00Z"
# PROJECT_PARAMS["publishedBefore"] = "2027-01-01T00:00:00Z"
# PROJECT_PARAMS["order"] = "viewCount"
# Metadata sections to retrieve for video IDs:
VIDEO_PARTS = "snippet,statistics,contentDetails"


#### Prepare the project collector
The next cell defines the provided collection functions. For video projects, they search for IDs and fetch each video’s metadata. For channel totals or comments, they use the corresponding endpoint and build the appropriate table. They also handle pagination, duplicate records and missing values.

**Your action:** run unchanged. **Output:** none expected; this cell defines functions. Collection starts in the following cell.

Comments include top-level text and reply counts, not reply text. Some videos disable comments. Present aggregate results rather than identifying quotes.


In [ ]:
# @title Project collector — run { display-mode: "form" }
def make_comment_table(payload, video_id):
    rows = []
    for thread in payload["items"]:
        outer = thread.get("snippet", {})
        comment = outer.get("topLevelComment", {})
        snippet = comment.get("snippet", {})
        rows.append({"comment_id": comment["id"], "video_id": outer.get("videoId", video_id),
                     "text": snippet.get("textDisplay"), "published_at": snippet.get("publishedAt"),
                     "likes": snippet.get("likeCount"), "reply_count": outer.get("totalReplyCount")})
    frame = pd.DataFrame(rows, columns=["comment_id", "video_id", "text", "published_at", "likes", "reply_count"])
    for field in ["likes", "reply_count"]:
        frame[field] = pd.to_numeric(frame[field], errors="raise").astype("Int64")
    frame["published_at"] = pd.to_datetime(frame["published_at"], utc=True)
    return frame.drop_duplicates("comment_id", keep="first").reset_index(drop=True)

def make_channel_table(payload):
    rows = []
    for item in payload["items"]:
        stats = item.get("statistics", {})
        rows.append({"channel_id": item["id"],
                     "title": item.get("snippet", {}).get("title"),
                     "subscribers": None if stats.get("hiddenSubscriberCount") else stats.get("subscriberCount"),
                     "videos": stats.get("videoCount"), "total_views": stats.get("viewCount")})
    frame = pd.DataFrame(rows, columns=["channel_id", "title", "subscribers", "videos", "total_views"])
    for field in ["subscribers", "videos", "total_views"]:
        frame[field] = pd.to_numeric(frame[field], errors="raise").astype("Int64")
    return frame

def collect_project(kind, parameters, max_pages=1):
    if kind not in {"channel_videos", "topic_videos", "channel_statistics", "video_comments"}:
        raise ValueError("Choose one of the four project kinds.")
    if not 1 <= max_pages <= 2:
        raise ValueError("Use one or two pages for this workshop.")
    params = {k: v for k, v in parameters.items() if k != "key"}
    if kind == "video_comments":
        if OFFLINE:
            raise ValueError("No comments are cached. Use a video project offline or enable live mode.")
        if not params.get("videoId") or params.get("part") != "snippet":
            raise ValueError("Use one videoId and part=snippet for comments.")
        if params.get("order", "time") not in {"time", "relevance"}:
            raise ValueError("Comment order must be time or relevance.")
        if not 1 <= int(params.get("maxResults", 20)) <= 20:
            raise ValueError("Use at most 20 comments per page.")
        params["textFormat"] = "plainText"
        params["fields"] = "nextPageToken,items(id,snippet(videoId,totalReplyCount,topLevelComment(id,snippet(textDisplay,publishedAt,likeCount))))"
        items, token, pages = [], None, 0
        base = {k: v for k, v in params.items() if k != "pageToken"}
        for _ in range(max_pages):
            query = base.copy()
            if token:
                query["pageToken"] = token
            try:
                data = api_get("commentThreads", query)
            except RuntimeError as exc:
                if "commentsDisabled" in str(exc):
                    raise RuntimeError("Comments are disabled for this video. Choose another video or another project.") from None
                raise
            items.extend(data["items"])
            pages += 1
            token = data.get("nextPageToken")
            if not token:
                break
        frame = make_comment_table({"items": items}, params["videoId"])
        return frame, {"unit": "top_level_comment", "parameters": params, "pages_read": pages,
                       "more_pages_available": bool(token), "records": len(frame),
                       "scope": "Top-level comments returned; reply text is not collected.", "coverage_verified": False}
    if kind == "channel_statistics":
        if OFFLINE:
            raise ValueError("No channel statistics are cached. Use a video project offline or enable live mode.")
        if not params.get("id") or len(params["id"].split(",")) > 20:
            raise ValueError("Use 1–20 verified channel IDs.")
        if not {"snippet", "statistics"}.issubset(set(params.get("part", "").split(","))):
            raise ValueError("Keep part=snippet,statistics for the channel table.")
        frame = make_channel_table(api_get("channels", params))
        return frame, {"unit": "channel", "parameters": params, "records": len(frame),
                       "requested_ids": params["id"].split(","),
                       "returned_ids": list(frame.channel_id), "coverage_verified": False}
    if params.get("type") != "video" or params.get("part") != "snippet":
        raise ValueError("Keep type=video and part=snippet in video search requests.")
    if kind == "channel_videos" and not params.get("channelId"):
        raise ValueError("The channel_videos template needs channelId.")
    if kind == "topic_videos" and not params.get("q"):
        raise ValueError("The topic_videos template needs a q search expression.")
    size = int(params.get("maxResults", 10))
    if not 1 <= size <= 20:
        raise ValueError("Use a page size from 1 to 20 for the project.")
    if OFFLINE:
        supported = {"part", "type", "channelId", "q", "order", "maxResults", "publishedAfter", "publishedBefore"}
        if set(params) - supported:
            raise ValueError("These extra parameters are not supported by the offline exercise.")
        if params.get("channelId", CHANNEL_ID) != "UC_dZp8bZipnjntBGLVHm6rw":
            raise ValueError("Offline data cover only the supplied example channel.")
        order = params.get("order", "date")
        if order not in {"date", "viewCount"}:
            raise ValueError("Offline mode supports date/viewCount, not relevance ranking.")
        records = list(FIXTURE["records"])
        if params.get("q"):
            records = [r for r in records if params["q"].casefold() in r["title"].casefold()]
        for field, lower in [("publishedAfter", True), ("publishedBefore", False)]:
            if params.get(field):
                boundary = pd.to_datetime(params[field], utc=True)
                records = [r for r in records if (pd.to_datetime(r["published_at"], utc=True) >= boundary
                           if lower else pd.to_datetime(r["published_at"], utc=True) < boundary)]
        ranked = sorted(records, key=lambda r: r["published_at"] if order == "date" else r["views"], reverse=True)
        ids = [r["video_id"] for r in ranked[:size * max_pages]]
        frame = make_table(offline_payload("videos", {"id": ",".join(ids)}))
        print("OFFLINE: local title/date filtering of saved records, not a replay of API search.")
        return frame, {"unit": "video", "parameters": params, "max_pages": max_pages,
                       "candidate_records": len(records), "records": len(frame),
                       "mode": "local_cache_selection", "coverage_verified": False}
    ids, token, pages = [], None, 0
    base_params = {k: v for k, v in params.items() if k != "pageToken"}
    for _ in range(max_pages):
        query = base_params.copy()
        if token:
            query["pageToken"] = token
        data = api_get("search", query)
        ids.extend(item["id"]["videoId"] for item in data["items"])
        ids = list(dict.fromkeys(ids))
        pages += 1
        token = data.get("nextPageToken")
        if not token:
            break
    items = []
    for start in range(0, len(ids), 50):
        data = api_get("videos", {"part": VIDEO_PARTS, "id": ",".join(ids[start:start + 50])})
        items.extend(data["items"])
    by_id = {item["id"]: item for item in items}
    frame = make_table({"items": [by_id[v] for v in ids if v in by_id]})
    return frame, {"unit": "video", "parameters": params, "metadata_parts": VIDEO_PARTS,
                   "pages_read": pages, "more_pages_available": bool(token), "records": len(frame),
                   "missing_metadata_ids": [v for v in ids if v not in by_id], "coverage_verified": False}


#### Run your collection
The next cell calls the collector with your chosen template, parameters and page limit. In live mode it sends the requests; offline mode selects from the saved records.

**Your action:** run after preparing the collector. **Output:** the number of collected records and a preview of up to ten rows. The full table is `project_df`; `project_info` records the collection settings and scope. Check that the rows match your intended unit: videos, channels or comments.


In [ ]:
project_df, project_info = collect_project(PROJECT_KIND, PROJECT_PARAMS, PROJECT_MAX_PAGES)
print("Collected:", len(project_df), "records")
print(project_df.head(10).to_string(index=False, max_colwidth=45))


### 5. Describe the results
The next cell checks row counts and unique IDs, reports missing values, and summarises available numeric columns. `count` means non-missing observations; `mean` is the average and `median` the middle value. Video projects also count returned videos by channel; comment projects summarise text length.

**Your action:** run unchanged, then choose one result that answers your question. For video views, compare the mean and median or identify the maximum in `project_df`. For comments, inspect `reply_count`; for channel totals, compare `videos` or `subscribers`.

**Output:** printed summaries. Describe these returned records. Search ranking affects selection; views are not unique people and subscriber counts may be hidden or rounded.


In [ ]:
identifier = "comment_id" if "comment_id" in project_df else "video_id" if "video_id" in project_df else "channel_id"
print("Rows:", len(project_df), "Unique IDs:", project_df[identifier].nunique())
missing = project_df.isna().sum()
if missing.any():
    print("Missing values:")
    print(missing[missing > 0])
numeric_fields = project_df.select_dtypes(include="number").columns.tolist()
numeric_data = project_df[numeric_fields].dropna(axis=1, how="all")
if not numeric_data.empty:
    print(numeric_data.agg(["count", "mean", "median", "min", "max"]).T)
if "video_id" in project_df and "channel_id" in project_df and not project_df.empty:
    print("Videos by channel:")
    print(project_df.groupby("channel_id", dropna=False).size().sort_values(ascending=False))
if "comment_id" in project_df:
    lengths = project_df["text"].astype("string").str.len().dropna()
    if not lengths.empty:
        print("Text length:")
        print(lengths.agg(["count", "mean", "median", "min", "max"]))


#### Optional: write your own plot
Use the collected table, `project_df`, to make one plot that helps answer your question. The next cell is yours to complete; no plotting code is provided.

**Your action:** choose a useful chart and write the code below. For example, compare views across selected videos, show how often channels appear in topic results, or plot the distribution of comment reply counts. Label the axes and add a clear title.

**Hints:** inspect `project_df.head()` and `project_df.columns`; select the relevant column(s), and aggregate first if your question needs counts by group. Use a plotting library you know. If needed, install matplotlib in a separate cell with `%pip install matplotlib`.

**Output:** your own chart when you write and run the code. Explain one pattern and one limit of the collected data. You can skip this cell and report a numerical summary instead.


In [ ]:
# Write your plotting code here using project_df.
# Choose a chart that answers your question; label its axes and title.
# Optional: leave this cell as it is to use the printed summary instead.


### 6. Save and summarise
The next cell creates a timestamped folder and saves `project_records.csv` plus `manifest.json`, which records your mode, request settings and collection scope. Live runs also save the JSON responses; offline runs save reconstructed example responses. The key is omitted from the saved request log.

**Your action:** run unchanged. In Colab’s left file browser, open `workshop_outputs/` and the printed run folder, then download your CSV and manifest. These files describe the data; they are not a complete archive of YouTube.

**Output:** the folder path. Prepare four lines: **your question, your request choices, one result, one limitation**. Sharing is optional, alone or in pairs, about two minutes. No slides needed.


In [ ]:
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir = OUTPUT_DIR / run_id
run_dir.mkdir(parents=True, exist_ok=False)
manifest = {
    "exercise_run_at": datetime.now(timezone.utc).isoformat(),
    "mode": "offline_teaching_fixtures" if OFFLINE else "live",
    "project_kind": PROJECT_KIND, "project": project_info,
    "guided_example": search_params, "requests": REQUEST_LOG,
    "original_observation_time": None if OFFLINE else "See received_at in api_responses.json",
    "offline_provenance": FIXTURE["provenance"] if OFFLINE else None,
    "scope": "Bounded chosen collection; no completeness or representativeness claim.",
}
project_df.to_csv(run_dir / "project_records.csv", index=False)
(run_dir / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
if not OFFLINE:
    (run_dir / "api_responses.json").write_text(json.dumps(RAW_RESPONSES, ensure_ascii=False, indent=2), encoding="utf-8")
else:
    (run_dir / "reconstructed_guided_payloads.json").write_text(
        json.dumps({"notice": "Reconstructed teaching responses, not originals", "search": search_data,
                    "videos": stats_data}, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved:", run_dir)


**Trying another request?** Edit the request-template cell, rerun it, then rerun the collection, summary, optional plot and save cells. `PROJECT_PARAMS` is a copy, so changing a request dictionary after that cell has run will not update the selected parameters until you rerun it.

For a second page, set `PROJECT_MAX_PAGES=2` in that cell. A page can contain fewer results than requested. Errors are not empty results: read the status/reason before retrying; a message about disabled comments means choose another video or project.


[Attribution and data provenance](https://github.com/manueltonneau/hpi-api/blob/main/SOURCES.md)
